## Installing Libraries

In [1]:
!pip install crewai crewai_tools openai langchain-openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 3.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.9/185.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.5/811.5 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

## Import Libraries and add OpenAI key

In [2]:
import sys  # Provides access to system-specific parameters and functions
import os  # Enables interaction with the operating system
from crewai import Agent, Task, Crew, Process  # Imports core classes for defining agents, tasks, workflows, and processes
from langchain_openai import ChatOpenAI  # Integration with OpenAI's language models via LangChain
from crewai_tools import WebsiteSearchTool  # A tool for performing website searches and extracting relevant data

In [3]:
from google.colab import userdata
OpenAI_API = userdata.get('OPENAI_API_KEY')  # Fetching the stored OpenAI API key from the Colab userdata storage
os.environ['OPENAI_API_KEY'] = OpenAI_API  # Setting the fetched API key as an environment variable for use in the application

## Initialize web search tool and Crew

In [4]:
web_tool = WebsiteSearchTool() # Initializing a WebsiteSearchTool instance for performing web searches and extracting relevant data

In [8]:
# Initialize a ChatOpenAI instance with a specific temperature and model
from crewai import LLM

llm = LLM(
    model="openai/gpt-4o-mini",
    temperature=0.1,
)
# verbose = True, enables detailed logging

# Function to generate a blog based on a given topic
async def generate_blog(topic):
    # Define the researcher agent responsible for gathering information
    researcher = Agent(
        role = "Internet Research",
        goal = f"""Conduct in-depth research on '{topic}' using reliable and diverse sources.
                 Identify key concepts, trends, examples, and statistics related to {topic}.
                 Your output should be a comprehensive, well-structured research document.""",
        verbose = True,
        llm=llm,
        backstory ="""You are a highly skilled researcher specializing in extracting, analyzing,
                     and consolidating information from the internet into actionable insights."""
    )

    # Define the content generator agent for drafting the blog
    generator = Agent(
        role='Content Generator',
        goal=f"""Draft a highly engaging and informative blog post on '{topic}'.
                 Include relevant examples, tutorials, visuals, and real-world applications.
                 Add emojis to make the blog reader-friendly. Use markdown for structure.""",
        verbose=True,
        allow_delegation=True,
        llm=llm,
        backstory="""You are an expert blogger known for creating detailed and captivating posts.
                     Your blogs are tutorial-like, enriched with real-world examples, and easy to follow."""
    )

    # Define the technical reviewer agent to ensure accuracy
    technical_reviewer = Agent(
        role="Technical Reviewer",
        goal=f"""Review the blog draft for technical accuracy, completeness, and relevance.
                 Ensure all examples, tutorials, and explanations are correct and useful.""",
        verbose=True,
        llm=llm,
        backstory="""You are a technical reviewer with expertise in validating and improving the
                     technical content of blogs, tutorials, and guides."""
    )

    # Define the copy editor agent for polishing the content
    copy_editor = Agent(
        role="Copy Editor",
        goal=f"""Polish the blog for grammar, style, and readability.
                 Ensure the content flows well and aligns with blogging best practices.""",
        verbose=True,
        llm=llm,
        backstory="""You are a skilled copy editor specializing in enhancing the quality, tone, and readability
                     of written content while preserving its original intent."""
    )

    # Define the markdown formatter agent for final formatting
    markdown_formatter = Agent(
        role="Markdown Formatter",
        goal=f"""Convert the finalized blog content into a properly structured markdown document.
                 Ensure the markdown formatting is clean, and includes proper headings, code blocks, and lists.""",
        verbose=True,
        llm=llm,
        backstory="""You are an expert in markdown formatting and structure, ensuring the content is clean,
                     organized, and ready for publishing."""
    )

    ## Define all tasks
    task_search = Task(
        description=f"""Perform extensive research on '{topic}' and compile a detailed,
                        easy-to-read research document (10,000 words).
                        Include examples, case studies, and all necessary data.""",
        expected_output=f"A well-organized research document about {topic}",
        max_inter=2,
        tools=[web_tool],
        agent=researcher
    )

    task_draft_blog = Task(
        description=f"""Use the research document to create a blog post (maximum 2,000 words).
                        Ensure the blog includes tutorials, code examples, real-world applications, and emojis.""",
        expected_output=f"A detailed and engaging blog post on {topic} in markdown format",
        agent=generator
    )

    task_review = Task(
        description=f"""Review the blog for technical accuracy and suggest any necessary improvements.
                        Ensure all examples, explanations, and tutorials are correct.""",
        expected_output=f"Feedback and revised content for technical correctness on {topic}",
        agent=technical_reviewer
    )

    task_edit = Task(
        description=f"""Edit the blog for grammar, style, and readability.
                        Ensure the content is polished and easy to read.""",
        expected_output=f"Polished and reader-friendly blog content for {topic}",
        agent=copy_editor
    )

    task_format_markdown = Task(
        description=f"""Format the finalized blog into a clean markdown .md document.
                        Ensure proper headings, lists, code blocks, and links are included.""",
        expected_output=f"A well-structured markdown file ready for publishing on {topic}",
        agent=markdown_formatter
    )

    # Define a crew to manage the agents and tasks
    crew = Crew(
        agents=[researcher, generator, technical_reviewer, copy_editor, markdown_formatter],
        tasks=[task_search, task_draft_blog, task_review, task_edit, task_format_markdown],
        verbose=True,
        process=Process.sequential)

    # Execute the process and return the result
    result = await crew.akickoff()
    return result

## Running the Crew

Note: All the red, green colored text during the generation of blog is the model running the processes internally. If you want to stop seeing it, put verbose=False everywhere.

In [9]:
# Define the topic for the blog
topic = "Bagging vs Boosting in Machine Learning"
# Call the generate_blog function to create the blog on the given topic
result = await generate_blog(topic)
# Print the final result of the blog generation process
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5a091be4-529a-49af-85ac-bd2c025d3b79                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Perform extensive research on 'Bagging vs Boosting in Machine Learning' and compile a detailed,          │
│                          easy-to-read research document (10,000 words).                                         │
│                          Include examples, case studies, and all necessary data.                                │
│  ID: 90a59486-9eb8-4ecc-b46b-e056fdc4f411                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Research                                                                                       │
│                                                                                                                 │
│  Task: Perform extensive research on 'Bagging vs Boosting in Machine Learning' and compile a detailed,          │
│                          easy-to-read research document (10,000 words).                                         │
│                          Include examples, case studies, and all necessary data.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_in_a_specific_website                                                                             │
│  Args: {'search_query': 'Bagging vs Boosting in Machine Learning', 'website': 'towardsdatascience.com'}         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_in_a_specific_website                                                                             │
│  Args: {'search_query': 'Bagging vs Boosting in Machine Learning', 'website': 'analyticsvidhya.com'}            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Markdown Formatter                                                                                      │
│                                                                                                                 │
│  Task: Format the finalized blog into a clean markdown .md document.                                            │
│                          Ensure proper headings, lists, code blocks, and links are included.                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Markdown Formatter                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```markdown                                                                                                    │
│  # Bagging vs Boosting in Machine Learning                                                                      │
│                                                                                                                 │
│  ## Table of Contents                                                                                           │
│                                                                                                                 │
│  1. **Introduction**                                                                                            │
│     - Overview of Ensemble Learning                                                                             │
│     - Importance of Bagging and Boosting                                                                        │
│                                                                                                                 │
│  2. **Bagging (Bootstrap Aggregating)**                                                                         │
│     - Definition and Concept                                                                                    │
│     - How Bagging Works                                                                                         │
│     - Advantages of Bagging                                                                                     │
│     - Disadvantages of Bagging                                                                                  │
│     - Popular Algorithms                                                                                        │
│       - Random Forest                                                                                           │
│     - Case Studies and Examples                                                                                 │
│                                                                                                                 │
│  3. **Boosting**                                                                                                │
│     - Definition and Concept                                                                                    │
│     - How Boosting Works                                                                                        │
│     - Advantages of Boosting                                                                                    │
│     - Disadvantages of Boosting                                                                                 │
│     - Popular Algorithms                                                                                        │
│       - AdaBoost                                                                                                │
│       - Gradient Boosting                                                                                       │
│       - XGBoost                                                                                                 │
│     - Case Studies and Examples                                                                                 │
│                                                                                                                 │
│  4. **Comparison of Bagging and Boosting**             

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Format the finalized blog into a clean markdown .md document.                                            │
│                          Ensure proper headings, lists, code blocks, and links are included.                    │
│  Agent: Markdown Formatter                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

```markdown
# Bagging vs Boosting in Machine Learning

## Table of Contents

1. **Introduction**
   - Overview of Ensemble Learning
   - Importance of Bagging and Boosting

2. **Bagging (Bootstrap Aggregating)**
   - Definition and Concept
   - How Bagging Works
   - Advantages of Bagging
   - Disadvantages of Bagging
   - Popular Algorithms
     - Random Forest
   - Case Studies and Examples

3. **Boosting**
   - Definition and Concept
   - How Boosting Works
   - Advantages of Boosting
   - Disadvantages of Boosting
   - Popular Algorithms
     - AdaBoost
     - Gradient Boosting
     - XGBoost
   - Case Studies and Examples

4. **Comparison of Bagging and Boosting**
   - Key Differences
   - Performance Metrics
   - Use Cases

5. **Conclusion**
   - Summary of Key Points
   - Future Trends in Ensemble Learning

6. **References**

---

## 1. Introduction

### Overview of Ensemble Learning

Ensemble learning is a powerful technique in machine learning that combines multiple models to 